# Session 4: Multi-Agent Workflows & Agentic Systems (Student Notebook)

Welcome to the capstone session of Day 2! In this notebook, you will learn how to design systems where multiple specialized agents collaborate — using supervisor patterns, agent handoffs, and parallel execution — to solve problems that are too complex for a single agent.

## Learning Objectives

By the end of this session, you will be able to:

1. **Design multi-agent architectures** using supervisor-worker patterns
2. **Implement agent handoffs** with context preservation
3. **Run agents in parallel** and merge their results
4. **Build collaborative agent teams** for complex tasks
5. **Integrate all Day 2 concepts** into an end-to-end system
6. **Evaluate trade-offs** between single-agent and multi-agent designs

In [ ]:
# ============================================================
# Imports and Setup
# ============================================================

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from typing import TypedDict, Literal
import json
import os

print("All imports successful!")

---
## Demos (Follow Along)

The following five demos are fully coded. Run each cell, observe the output, and make sure you understand what is happening before moving on.

### Demo 1: Supervisor-Worker Pattern with LangGraph

The supervisor pattern is the most common multi-agent architecture. A **supervisor** agent receives the task, decides which **worker** agent should handle it, and routes accordingly. The supervisor may also combine results from multiple workers.

In [ ]:
# Demo 1 - Supervisor-Worker Pattern

class SupervisorState(TypedDict):
    task: str
    assigned_to: str
    worker_output: str
    final_response: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def supervisor_node(state: SupervisorState) -> dict:
    """Supervisor decides which worker should handle the task."""
    response = llm.invoke([
        SystemMessage(content=(
            "You are a supervisor. Assign the task to one worker: "
            "'researcher' (for questions needing facts/analysis), "
            "'writer' (for content creation), or "
            "'coder' (for programming tasks). "
            "Respond with just the worker name."
        )),
        HumanMessage(content=state["task"])
    ])
    assigned = response.content.strip().lower()
    print(f"  Supervisor assigned to: {assigned}")
    return {"assigned_to": assigned}

def researcher_node(state: SupervisorState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are a research analyst. Provide factual, well-structured analysis."),
        HumanMessage(content=state["task"])
    ])
    return {"worker_output": f"[Research] {response.content}"}

def writer_node(state: SupervisorState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are a creative writer. Write engaging, polished content."),
        HumanMessage(content=state["task"])
    ])
    return {"worker_output": f"[Writer] {response.content}"}

def coder_node(state: SupervisorState) -> dict:
    response = llm.invoke([
        SystemMessage(content="You are an expert programmer. Write clean, documented code."),
        HumanMessage(content=state["task"])
    ])
    return {"worker_output": f"[Coder] {response.content}"}

def review_node(state: SupervisorState) -> dict:
    """Supervisor reviews and finalizes the worker output."""
    response = llm.invoke([
        SystemMessage(content="You are a supervisor reviewing work. Add a brief quality assessment and finalize."),
        HumanMessage(content=f"Task: {state['task']}\n\nWorker output:\n{state['worker_output']}")
    ])
    return {"final_response": response.content}

def route_to_worker(state: SupervisorState) -> str:
    a = state["assigned_to"]
    if "researcher" in a: return "researcher"
    if "writer" in a: return "writer"
    return "coder"

# Build graph
graph = StateGraph(SupervisorState)
graph.add_node("supervisor", supervisor_node)
graph.add_node("researcher", researcher_node)
graph.add_node("writer", writer_node)
graph.add_node("coder", coder_node)
graph.add_node("review", review_node)

graph.set_entry_point("supervisor")
graph.add_conditional_edges("supervisor", route_to_worker, {
    "researcher": "researcher", "writer": "writer", "coder": "coder"
})
graph.add_edge("researcher", "review")
graph.add_edge("writer", "review")
graph.add_edge("coder", "review")
graph.add_edge("review", END)

app = graph.compile()

# Test
for task in ["Explain the pros and cons of microservices", "Write a haiku about programming", "Write a Python function to merge two sorted lists"]:
    print(f"\nTask: {task}")
    result = app.invoke({"task": task, "assigned_to": "", "worker_output": "", "final_response": ""})
    print(f"Response: {result['final_response'][:200]}...")

### Demo 2: Agent-to-Agent Handoff and Context Passing

In a handoff, one agent transfers control to another along with relevant context. The receiving agent picks up where the first left off. This is useful when a task requires different expertise at different stages.

In [ ]:
# Demo 2 - Agent-to-Agent Handoff

class HandoffState(TypedDict):
    user_request: str
    draft: str
    review_notes: str
    final_output: str
    handoff_context: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

def drafting_agent(state: HandoffState) -> dict:
    """Agent 1: Creates the initial draft."""
    response = llm.invoke([
        SystemMessage(content="You are a content drafter. Create a draft and note any areas you're unsure about."),
        HumanMessage(content=state["user_request"])
    ])
    context = f"Draft completed. Areas to review: accuracy of technical claims, tone consistency."
    print(f"  [Drafter] Created draft ({len(response.content)} chars)")
    return {"draft": response.content, "handoff_context": context}

def review_agent(state: HandoffState) -> dict:
    """Agent 2: Reviews the draft (receives handoff context)."""
    response = llm.invoke([
        SystemMessage(content="You are a content reviewer. Review the draft and provide specific improvement notes."),
        HumanMessage(content=f"Handoff notes: {state['handoff_context']}\n\nDraft to review:\n{state['draft']}")
    ])
    print(f"  [Reviewer] Notes: {response.content[:80]}...")
    return {"review_notes": response.content}

def editing_agent(state: HandoffState) -> dict:
    """Agent 3: Produces the final output based on draft + review."""
    response = llm.invoke([
        SystemMessage(content="You are an editor. Apply the review notes to improve the draft. Output the final version."),
        HumanMessage(content=f"Original draft:\n{state['draft']}\n\nReview notes:\n{state['review_notes']}")
    ])
    print(f"  [Editor] Finalized ({len(response.content)} chars)")
    return {"final_output": response.content}

graph = StateGraph(HandoffState)
graph.add_node("drafter", drafting_agent)
graph.add_node("reviewer", review_agent)
graph.add_node("editor", editing_agent)
graph.set_entry_point("drafter")
graph.add_edge("drafter", "reviewer")
graph.add_edge("reviewer", "editor")
graph.add_edge("editor", END)
app = graph.compile()

result = app.invoke({
    "user_request": "Write a brief explanation of how neural networks learn",
    "draft": "", "review_notes": "", "final_output": "", "handoff_context": ""
})
print(f"\nFinal output:\n{result['final_output'][:300]}...")

### Demo 3: Parallel Agent Execution

Some tasks can be split into independent subtasks that run in parallel. Each agent works on its part simultaneously, and a merger step combines the results. This reduces total execution time.

In [ ]:
# Demo 3 - Parallel Agent Execution (Simulated)

# Note: True parallel execution requires async. This demo simulates the pattern.

class ParallelState(TypedDict):
    topic: str
    pros_analysis: str
    cons_analysis: str
    examples_analysis: str
    merged_report: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

def pros_agent(state: ParallelState) -> dict:
    r = llm.invoke([SystemMessage(content="List 3 key advantages. Be specific."), HumanMessage(content=state["topic"])])
    print(f"  [Pros Agent] Done")
    return {"pros_analysis": r.content}

def cons_agent(state: ParallelState) -> dict:
    r = llm.invoke([SystemMessage(content="List 3 key disadvantages. Be specific."), HumanMessage(content=state["topic"])])
    print(f"  [Cons Agent] Done")
    return {"cons_analysis": r.content}

def examples_agent(state: ParallelState) -> dict:
    r = llm.invoke([SystemMessage(content="Give 3 real-world examples or use cases."), HumanMessage(content=state["topic"])])
    print(f"  [Examples Agent] Done")
    return {"examples_analysis": r.content}

def merge_agent(state: ParallelState) -> dict:
    r = llm.invoke([
        SystemMessage(content="Combine these analyses into a balanced, well-structured report."),
        HumanMessage(content=f"Topic: {state['topic']}\n\nPros:\n{state['pros_analysis']}\n\nCons:\n{state['cons_analysis']}\n\nExamples:\n{state['examples_analysis']}")
    ])
    return {"merged_report": r.content}

# Build graph — fan-out / fan-in pattern
graph = StateGraph(ParallelState)
graph.add_node("pros", pros_agent)
graph.add_node("cons", cons_agent)
graph.add_node("examples", examples_agent)
graph.add_node("merge", merge_agent)

# Fan-out: entry connects to all three agents
graph.set_entry_point("pros")
graph.add_edge("pros", "cons")
graph.add_edge("cons", "examples")
# Fan-in: all three connect to merge
graph.add_edge("examples", "merge")
graph.add_edge("merge", END)

app = graph.compile()

result = app.invoke({
    "topic": "Using LLMs for automated code review",
    "pros_analysis": "", "cons_analysis": "", "examples_analysis": "", "merged_report": ""
})
print(f"\nMerged Report:\n{result['merged_report'][:400]}...")

### Demo 4: Collaborative Writing with Specialized Agents

Multiple agents contribute different aspects to a single document. An outliner creates the structure, a writer fills in the content, and a stylist polishes the language.

In [ ]:
# Demo 4 - Collaborative Writing

class CollabState(TypedDict):
    topic: str
    outline: str
    draft: str
    polished: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

def outliner_agent(state: CollabState) -> dict:
    r = llm.invoke([
        SystemMessage(content="You are an outliner. Create a clear 3-section outline with bullet points for each section."),
        HumanMessage(content=f"Create an outline for: {state['topic']}")
    ])
    print(f"  [Outliner] Created outline")
    return {"outline": r.content}

def writer_agent(state: CollabState) -> dict:
    r = llm.invoke([
        SystemMessage(content="You are a content writer. Write the full text following this outline. Keep it concise."),
        HumanMessage(content=f"Topic: {state['topic']}\n\nOutline:\n{state['outline']}")
    ])
    print(f"  [Writer] Wrote draft ({len(r.content)} chars)")
    return {"draft": r.content}

def stylist_agent(state: CollabState) -> dict:
    r = llm.invoke([
        SystemMessage(content="You are a style editor. Polish the text for clarity, flow, and engagement. Maintain the same structure."),
        HumanMessage(content=state["draft"])
    ])
    print(f"  [Stylist] Polished ({len(r.content)} chars)")
    return {"polished": r.content}

graph = StateGraph(CollabState)
graph.add_node("outliner", outliner_agent)
graph.add_node("writer", writer_agent)
graph.add_node("stylist", stylist_agent)
graph.set_entry_point("outliner")
graph.add_edge("outliner", "writer")
graph.add_edge("writer", "stylist")
graph.add_edge("stylist", END)
app = graph.compile()

result = app.invoke({"topic": "Getting started with LangGraph for AI agents", "outline": "", "draft": "", "polished": ""})
print(f"\nPolished article:\n{result['polished'][:500]}...")

### Demo 5: Building an End-to-End Multi-Agent System

This demo brings together supervisor routing, specialized workers, and quality review into a complete system that handles diverse user requests.

In [ ]:
# Demo 5 - End-to-End Multi-Agent System

class E2EState(TypedDict):
    user_input: str
    intent: str
    agent_output: str
    quality_score: str
    final_output: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def intent_classifier(state: E2EState) -> dict:
    r = llm.invoke([
        SystemMessage(content="Classify intent: 'question', 'task', or 'chat'. One word."),
        HumanMessage(content=state["user_input"])
    ])
    intent = r.content.strip().lower()
    print(f"  Intent: {intent}")
    return {"intent": intent}

def question_agent(state: E2EState) -> dict:
    r = llm.invoke([SystemMessage(content="Answer the question accurately and concisely."), HumanMessage(content=state["user_input"])])
    return {"agent_output": r.content}

def task_agent(state: E2EState) -> dict:
    r = llm.invoke([SystemMessage(content="Complete the task step by step."), HumanMessage(content=state["user_input"])])
    return {"agent_output": r.content}

def chat_agent(state: E2EState) -> dict:
    r = llm.invoke([SystemMessage(content="Be friendly and conversational."), HumanMessage(content=state["user_input"])])
    return {"agent_output": r.content}

def quality_check(state: E2EState) -> dict:
    r = llm.invoke([
        SystemMessage(content="Rate response quality 1-10. Output JSON: {\"score\": N, \"feedback\": \"...\"}"),
        HumanMessage(content=f"User asked: {state['user_input']}\n\nResponse: {state['agent_output']}")
    ])
    return {"quality_score": r.content, "final_output": state["agent_output"]}

def route_intent(state: E2EState) -> str:
    if "question" in state["intent"]: return "question"
    if "task" in state["intent"]: return "task"
    return "chat"

graph = StateGraph(E2EState)
graph.add_node("classify", intent_classifier)
graph.add_node("question", question_agent)
graph.add_node("task", task_agent)
graph.add_node("chat", chat_agent)
graph.add_node("quality", quality_check)

graph.set_entry_point("classify")
graph.add_conditional_edges("classify", route_intent, {"question": "question", "task": "task", "chat": "chat"})
graph.add_edge("question", "quality")
graph.add_edge("task", "quality")
graph.add_edge("chat", "quality")
graph.add_edge("quality", END)
app = graph.compile()

for inp in ["What is the capital of Japan?", "Write a sorting function", "Hey, how's it going?"]:
    print(f"\nUser: {inp}")
    r = app.invoke({"user_input": inp, "intent": "", "agent_output": "", "quality_score": "", "final_output": ""})
    print(f"Response: {r['final_output'][:150]}...")
    print(f"Quality: {r['quality_score'][:80]}")

---
## Tasks (Your Turn!)

Now it is your turn to practice. Each task below has a description, hints, and a code skeleton with `TODO` placeholders.

### Task 1: Build a Two-Agent Supervisor-Worker System

Create a system with a supervisor that routes tasks to either an "analyst" (for data questions) or a "creative" (for content tasks), then reviews the output.

In [ ]:
# Task 1 - Build a Two-Agent Supervisor-Worker System

# TODO: Create a supervisor-worker system with:
# 1. supervisor_node: Classifies task as 'analyst' or 'creative'
# 2. analyst_node: Handles data/analysis questions with a factual persona
# 3. creative_node: Handles content creation with a creative persona
# 4. review_node: Supervisor reviews and approves the output
#
# Hint: Use conditional edges after the supervisor
# Hint: Both workers should connect to the review node
# Hint: State needs: task, assigned_to, worker_output, final_response

# YOUR CODE HERE


# Test
# result = app.invoke({"task": "Analyze the trend of AI adoption in healthcare", ...})
# print(result["final_response"])

### Task 2: Implement Agent Handoff with Context Preservation

Build a three-agent pipeline where each agent passes structured context to the next: researcher -> analyst -> presenter.

In [ ]:
# Task 2 - Implement Agent Handoff with Context Preservation

# TODO: Build a handoff pipeline with:
# 1. researcher_node: Gathers raw information about the topic
# 2. analyst_node: Receives research + handoff context, produces insights
# 3. presenter_node: Receives insights + handoff context, creates a presentation summary
#
# Each node should:
# - Read the handoff_context from the previous agent
# - Update the handoff_context with notes for the next agent
#
# Hint: handoff_context should be a string that accumulates across agents
# Hint: Each agent adds its notes to the context before passing it on

# YOUR CODE HERE


# Test
# result = app.invoke({"topic": "The future of remote work", ...})
# print(result["presentation"])

### Task 3: Create a Parallel Research and Synthesis Pipeline

Build a system where three research agents explore different aspects of a topic (technical, business, social), then a synthesis agent combines their findings.

In [ ]:
# Task 3 - Create a Parallel Research and Synthesis Pipeline

# TODO: Build a fan-out/fan-in pipeline with:
# 1. technical_agent: Researches technical aspects
# 2. business_agent: Researches business/market aspects
# 3. social_agent: Researches social/cultural impact
# 4. synthesis_agent: Combines all three into a unified report
#
# Hint: State should have fields for each agent's output + the merged report
# Hint: In a sequential graph, run agents one after another
# Hint: The synthesis agent should reference all three outputs

# YOUR CODE HERE


# Test
# result = app.invoke({"topic": "Generative AI in education", ...})
# print(result["report"])

### Task 4: Design and Document a Complete Multi-Agent Architecture

Design a multi-agent system for a real-world use case of your choice. Implement a simplified version and document the architecture.

In [ ]:
# Task 4 - Design and Document a Complete Multi-Agent Architecture

# TODO: Pick a use case and build a multi-agent system.
# Suggested use cases:
#   - Customer support (triage -> specialist -> quality check)
#   - Content pipeline (research -> write -> edit -> publish)
#   - Code review (analyze -> security check -> style check -> report)
#
# Requirements:
# 1. At least 3 specialized agents
# 2. At least one conditional edge
# 3. A final output that combines multiple agent contributions
# 4. Print a description of your architecture before running it
#
# Hint: Start by defining your state schema
# Hint: Draw the graph on paper first
# Hint: Test with at least 2 different inputs

# YOUR CODE HERE


# Document your architecture
# print("Architecture: ...")
# print("Agents: ...")
# print("Flow: ...")

---
## Summary

In this capstone session you learned how to build multi-agent systems:

1. **Supervisor-Worker Pattern** -- A central supervisor routes tasks to specialized worker agents based on task type.
2. **Agent Handoffs** -- Agents pass structured context to each other, enabling sequential specialization.
3. **Parallel Execution** -- Independent subtasks run concurrently, with a merger step combining results.
4. **Collaborative Writing** -- Multiple agents contribute different aspects to a shared output.
5. **End-to-End Systems** -- Complete architectures combining classification, specialized agents, and quality review.

**Day 2 Complete!** You now have the skills to build production-grade agentic systems using OpenAI structured outputs, LangChain chains and tools, LangGraph workflows, and multi-agent architectures.